In [29]:
import wandb
from torch import nn
import typing
import torchvision
from xaikd import datasets
import torch
from collections import OrderedDict

In [25]:
ARTIFACT_DIR = "./tmp"

In [12]:

def model_generator(arch: str) -> typing.Optional[nn.Module]:
    num_outputs = datasets.celeba.NUM_CELEBA_ATTRIBUTES
    model = None
    if arch == "resnet18":
        model = torchvision.models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_outputs)
    elif arch == "resnet50":
        model = torchvision.models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_outputs)
    elif arch == "vitb16":
        model = torchvision.models.vit_b_16(
            weights=None
        )
        model.heads.head = nn.Linear(model.hidden_dim, num_outputs)
    elif arch == "wideresnet50-2":
        model = torchvision.models.wide_resnet50_2(
            weights=None
        )
        model.fc = nn.Linear(model.fc.in_features, num_outputs)
    else:
        raise ValueError(f"{arch} doesn't exist")

    return model


In [20]:
def load_model_from_artifact(
    model_template_object: nn.Module,
    wandb_run_path: str,
    wandb_artifact_suffix="best",
    model_attribute="encoder",
) -> nn.Module:

    slugs = wandb_run_path.split("/")

    assert len(slugs) == 3

    wandb_project = "/".join(slugs[:2])
    wandb_runid = slugs[-1]

    agent = wandb.Api()

    artifact: wandb.Artifact = agent.artifact(
        f"{wandb_project}/model-{wandb_runid}:{wandb_artifact_suffix}"
    )

    artifact_dir = artifact.download(root=ARTIFACT_DIR)

    ckpt = torch.load(
        f"{artifact_dir}/model.ckpt",
        map_location=torch.device("cpu"),
        weights_only=False,
    )

    state_dict = ckpt["state_dict"]

    student_state_dict = dict()

    for key in state_dict.keys():
        if key.split(".")[0] == model_attribute:
            student_state_dict[key] = state_dict[key]

    trainer_wrapper = nn.Sequential(
        OrderedDict([(model_attribute, model_template_object)])
    )
    trainer_wrapper.load_state_dict(student_state_dict)

    return model_template_object

In [33]:
def load_model(arch, run_path):
    
    model = model_generator(arch)
    model = load_model_from_artifact(model, wandb_run_path=run_path)
    return model

for arch, run_path in [
    ("resnet18", "p16i/xaikd-training-teacher-models/zbgow8eu"),
    ("resnet50", "p16i/xaikd-training-teacher-models/cuoynabf"),
    ("wideresnet50-2", "p16i/xaikd-training-teacher-models/t0sg5wcp"),
    ("vitb16", "p16i/xaikd-training-teacher-models/6ttr2icx"),
]: 
    model = load_model(arch=arch, run_path=run_path)
    slug = arch + "--" + run_path.replace("/", "-")
    torch.save(model.state_dict(), f"{ARTIFACT_DIR}/output/{slug}")

wandb:   1 of 1 files downloaded.  
wandb: Downloading large artifact model-cuoynabf:best, 90.30MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:4.9
wandb: Downloading large artifact model-t0sg5wcp:best, 255.63MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:11.1
wandb: Downloading large artifact model-6ttr2icx:best, 327.48MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:13.3
